In [45]:
import torch
import os
import pandas as pd
from torch import optim
from torchvision import models
from pytorch_memlab import LineProfiler
from pytorch_memlab import MemReporter     
from model import init_model
from torchinfo import summary
from pathlib import Path
from pytorch_memlab import profile



In [46]:
ROOT_DIR = Path(os.path.dirname(os.path.abspath('')))
TOTAL_TRAIN_SAMPLES = 50
TOTAL_TEST_SAMPLES = 10
CHANNELS = 3
BATCH_SIZE = 16
DEVICE = "cuda:1"
OCT_PRESENCE = "Usando OCT"
DUAL_IMAGE = "Dual Image"
DATA_PATH = "../data.csv"
SUMMARY_PATH = "../model_summary.csv"
HISTORY_PATH = "../history_csv"
FT_SIZE = 24
OUTPUT_TAB = 5

In [47]:
model = models.regnet_x_1_6gf(True)
num_ftrs = model.fc.in_features
model.fc = torch.nn.Linear(num_ftrs, 1)
torch.cuda.empty_cache()

model.to("cuda")
#reporter = MemReporter(model)
#reporter.report(verbose=True)

/data/home/felipemarcelino/google_drive/comp_science/projects/glaucoma/venv/lib/python3.9/site-packages/torchvision/models/_utils.py:135: UserWarning: Using 'weights' as positional parameter(s) is deprecated since 0.13 and will be removed in 0.15. Please use keyword parameter(s) instead.
  warnings.warn(
/data/home/felipemarcelino/google_drive/comp_science/projects/glaucoma/venv/lib/python3.9/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and will be removed in 0.15. The current behavior is equivalent to passing `weights=RegNet_X_1_6GF_Weights.IMAGENET1K_V1`. You can also use `weights=RegNet_X_1_6GF_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


RegNet(
  (stem): SimpleStemIN(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
  )
  (trunk_output): Sequential(
    (block1): AnyStage(
      (block1-0): ResBottleneckBlock(
        (proj): Conv2dNormActivation(
          (0): Conv2d(32, 72, kernel_size=(1, 1), stride=(2, 2), bias=False)
          (1): BatchNorm2d(72, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        )
        (f): BottleneckTransform(
          (a): Conv2dNormActivation(
            (0): Conv2d(32, 72, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (1): BatchNorm2d(72, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): ReLU(inplace=True)
          )
          (b): Conv2dNormActivation(
            (0): Conv2d(72, 72, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=3, bias=False)
            

In [48]:
@profile
def test_memory():
    for i in range(2):
        i = 0
        torch.cuda.empty_cache()
        target = torch.randint(0,2,(16,1)).float().to("cuda")
        sample = torch.rand(16,3,224,224).to("cuda")
        params_to_update = model.parameters()
        for name, param in model.named_parameters():
            param.requires_grad = True

        optimizer = optim.SGD(params_to_update, lr=0.01)
        criterion =  torch.nn.BCEWithLogitsLoss()

        # compute output
    #     if i < 1:
    #         torch.cuda.reset_peak_memory_stats()
        output = model(sample)
        loss = criterion(output, target)
        if i < 1:
            print('Max CUDA memory allocated on forward: ',(torch.cuda.max_memory_allocated() + torch.cuda.max_memory_reserved()) / 10**6)

        # measure accuracy and record loss
        #acc1, acc5 = accuracy(output, target)
        #losses.update(loss.detach().item(), images.size(0))
        #top1.update(acc1[0], images.size(0))
        #top5.update(acc5[0], images.size(0))

        # compute gradient and do SGD step
    #     if i < 1:
    #         torch.cuda.reset_peak_memory_stats()
        optimizer.zero_grad()
        if i < 1:
            print('Max CUDA memory allocated on backward: ', torch.cuda.max_memory_allocated() / 10**6)    
        loss.backward()
        optimizer.step()
    #     if i < 1:
    #         print('Max CUDA memory allocated on backward: ', torch.cuda.max_memory_allocated() / 10**6)

In [42]:
with LineProfiler(test_memory) as prof:
    torch.cuda.empty_cache()
    test_memory()

df_peak = pd.read_html(prof.display()._repr_html_())[0]

Max CUDA memory allocated on forward:  3491.2896
Max CUDA memory allocated on backward:  1708.7104
Max CUDA memory allocated on forward:  3559.540736
Max CUDA memory allocated on backward:  1745.504256


/data/home/felipemarcelino/google_drive/comp_science/projects/glaucoma/venv/lib/python3.9/site-packages/pytorch_memlab/line_profiler/line_records.py:60: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  records = (_accumulate_line_records(raw_line_records)
/data/home/felipemarcelino/google_drive/comp_science/projects/glaucoma/venv/lib/python3.9/site-packages/pytorch_memlab/line_profiler/line_records.py:215: FutureWarning: this method is deprecated in favour of `Styler.hide(axis="index")`
  html[qual_name] = (style
/data/home/felipemarcelino/google_drive/comp_science/projects/glaucoma/venv/lib/python3.9/site-packages/pytorch_memlab/line_profiler/line_records.py:215: FutureWarning: this method is deprecated in favour of `Styler.to_html()`
  html[qual_name] = (style


In [36]:
df_peak.columns = df_peak.columns.droplevel([1,2])

In [37]:
df_peak

,active_bytes,reserved_bytes,line,code
0,606.96M,1.34G,1,@profile
1,NaN,NaN,2,def test_memory():
2,650.07M,1.63G,3,for i in range(2):
3,649.49M,1.62G,4,i = 0
4,649.49M,1.62G,5,torch.cuda.empty_cache()
5,649.49M,1.37G,6,"target = torch.randint(0,2,(16,1)).float().to(..."
6,659.25M,1.37G,7,"sample = torch.rand(16,3,224,224).to(""cuda"")"
7,650.07M,1.37G,8,params_to_update = model.parameters()
8,650.07M,1.37G,9,"for name, param in model.named_parameters():"
9,650.07M,1.37G,10,param.requires_grad = True
